# Một giải pháp kinh doanh đầy đủ

## Bây giờ chúng ta sẽ nâng dự án Day 1 lên mức tiếp theo

### THÁCH THỨC KINH DOANH:

Tạo một sản phẩm dựng Brochure (tờ giới thiệu) cho một công ty, dùng cho khách hàng tiềm năng, nhà đầu tư và ứng viên tiềm năng.

Chúng ta sẽ được cung cấp tên công ty và website chính của họ.

Xem cuối notebook này để có ví dụ ứng dụng kinh doanh thực tế.

Và nhớ: tôi luôn sẵn sàng nếu bạn gặp vấn đề hoặc có ý tưởng! Hãy liên hệ.

In [7]:
# imports (nhập thư viện)
# Nếu các dòng này thất bại, hãy kiểm tra bạn đang chạy trong môi trường đã 'activated' (kích hoạt) với (llms) trên command prompt (dòng lệnh)

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [8]:
# Khởi tạo và các constants (hằng số)

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key trông ổn đến giờ")
else:
    print("Có thể có vấn đề với API key của bạn? Hãy xem notebook troubleshooting!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key trông ổn đến giờ


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## Bước đầu tiên: Để GPT-5-nano tìm ra những link (liên kết) nào là relevant (liên quan)

### Dùng một lời gọi gpt-5-nano để đọc các link trên webpage (trang web), rồi trả lời bằng JSON có cấu trúc.  
Nó nên quyết định link nào là relevant, và thay relative links (liên kết tương đối) như "/about" bằng "https://company.com/about".  
Chúng ta sẽ dùng "one shot prompting" (prompt một ví dụ) — cung cấp một example (ví dụ) về cách nó nên trả lời ngay trong prompt.

Đây là một use case (tình huống sử dụng) xuất sắc cho LLM, vì nó đòi hỏi hiểu biết tinh tế. Hãy tưởng tượng phải code việc này mà không có LLM, bằng cách parse (phân tích cú pháp) và phân tích webpage — sẽ rất khó!

Sidenote (ghi chú thêm): có một kỹ thuật nâng cao hơn gọi là "Structured Outputs" (đầu ra có cấu trúc), trong đó chúng ta yêu cầu model trả lời theo một spec (đặc tả). Chúng ta sẽ học kỹ thuật này ở Week 8 trong dự án Agentic AI tự chủ.

In [9]:
link_system_prompt = """
Bạn được cung cấp một list of links tìm thấy trên một webpage.
Bạn có thể quyết định những links nào relevant nhất để đưa vào brochure về công ty,
ví dụ links tới About page, hoặc Company page, hoặc Careers/Jobs pages.
Bạn nên respond bằng JSON như example này:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [10]:
def get_links_user_prompt(url):
    user_prompt = f"""
Đây là list of links trên website {url} -
Hãy quyết định những web links nào relevant cho một brochure về công ty,
respond với full https URL ở JSON format.
Không bao gồm Terms of Service, Privacy, email links.

Links (một số có thể là relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://edwarddonner.com"))


Đây là list of links trên website https://edwarddonner.com -
Hãy quyết định những web links nào relevant cho một brochure về công ty,
respond với full https URL ở JSON format.
Không bao gồm Terms of Service, Privacy, email links.

Links (một số có thể là relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-a

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [13]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'curriculum / resume',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient / skills',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page - project: connect four',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page - project: outsmart',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog / posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'external product / Nebula page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https:/

In [14]:
def select_relevant_links(url):
    print(f"Đang chọn relevant links cho {url} bằng cách gọi {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Tìm thấy {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [15]:
select_relevant_links("https://huggingface.co")

Đang chọn relevant links cho https://huggingface.co bằng cách gọi gpt-5-nano
Tìm thấy 10 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

## Bước thứ hai: làm brochure (tờ giới thiệu)!

Ghép tất cả chi tiết vào một prompt khác gửi tới GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Đang chọn relevant links cho https://huggingface.co bằng cách gọi gpt-5-nano
Tìm thấy 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
×
We are happy to share our intention to join forces with
NVIDIA
.
Read the announcement
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
XHToken/Spark-X2.5-4B
Updated
5 days ago
•
7.22k
•
744
Qwen/Qwen3.8-27B
Updated
25 days ago
•
6.42M
•
14.3k
google/timesfm-3.0-pytorch
Updated
5 days ago
•
272k
•
580
ISTA-DASLab/Qwen3.8-27B

In [18]:
brochure_system_prompt = """
Bạn là một assistant phân tích contents của vài trang relevant từ website công ty
và tạo một brochure ngắn về công ty cho khách hàng tiềm năng, nhà đầu tư và ứng viên.
Respond bằng markdown, không dùng code blocks.
Bao gồm chi tiết về company culture, customers và careers/jobs nếu bạn có thông tin.
"""

# Hoặc bỏ comment các dòng dưới để có brochure hài hước hơn — điều này cho thấy việc đưa 'tone' (giọng điệu) vào dễ đến mức nào:

# brochure_system_prompt = """
# Bạn là một assistant phân tích contents của vài trang relevant từ website công ty
# và tạo một brochure ngắn, hài hước, giải trí, dí dỏm về công ty cho khách hàng tiềm năng, nhà đầu tư và ứng viên.
# Respond bằng markdown, không dùng code blocks.
# Bao gồm chi tiết về company culture, customers và careers/jobs nếu bạn có thông tin.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
Bạn đang xem một công ty tên: {company_name}
Đây là contents của landing page và các trang relevant khác;
dùng thông tin này để dựng một brochure ngắn về công ty bằng markdown, không dùng code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate (cắt ngắn) nếu hơn 5.000 characters (ký tự)
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Đang chọn relevant links cho https://huggingface.co bằng cách gọi gpt-5-nano
Tìm thấy 12 relevant links


'\nBạn đang xem một công ty tên: HuggingFace\nĐây là contents của landing page và các trang relevant khác;\ndùng thông tin này để dựng một brochure ngắn về công ty bằng markdown, không dùng code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\n×\nWe are happy to share our intention to join forces with\nNVIDIA\n.\nRead the announcement\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nXHToken/Spark-X2.5-4B\nUpdated\n5 da

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Đang chọn relevant links cho https://huggingface.co bằng cách gọi gpt-5-nano
Tìm thấy 14 relevant links


# Brochure Giới Thiệu Công Ty Hugging Face

---

## Giới Thiệu Chung

**Hugging Face** là cộng đồng AI hàng đầu, xây dựng tương lai của trí tuệ nhân tạo (AI) thông qua nền tảng hợp tác mở dành cho cộng đồng machine learning trên toàn thế giới. Công ty cung cấp một môi trường để tạo, khám phá và cộng tác trên các mô hình, bộ dữ liệu (datasets) và các ứng dụng AI đa dạng.

- Nền tảng hỗ trợ hơn **2 triệu mô hình AI** và hơn **500,000 datasets**.
- Không gian hợp tác để phát triển, chạy thử và triển khai các ứng dụng AI (Spaces).
- Đang hợp tác chiến lược với "ông lớn" công nghệ NVIDIA nhằm thúc đẩy đổi mới trong lĩnh vực AI.

Nền tảng của Hugging Face là điểm đến cho các chuyên gia, nhà nghiên cứu và doanh nghiệp muốn cùng nhau phát triển và áp dụng AI một cách mở và hiệu quả.

---

## Sản Phẩm & Dịch Vụ

### 1. Models (Mô hình AI)
- Cung cấp thư viện mô hình AI phong phú, cập nhật thường xuyên với số lượng tải xuống và tương tác rất lớn.
- Các mô hình nổi bật như Qwen3.8-27B, Spark-X2.5-4B, TimesFM, LTX-2.5...

### 2. Datasets (Bộ dữ liệu)
- Kho dữ liệu đa dạng và có chất lượng cao với hơn 500k datasets cho các ứng dụng học máy.
- Ví dụ: tiktok-videos-4b, IMDB, SQuAD,...

### 3. Spaces (Không gian ứng dụng)
- Nơi chạy và giới thiệu các ứng dụng AI sáng tạo.
- Cung cấp sẵn các sandbox để thử nghiệm và demo các công cụ mô hình.

### 4. Enterprise Solution
- Dịch vụ hỗ trợ doanh nghiệp với các gói PRO, Inference Endpoints, Storage Buckets, giúp tích hợp mô hình AI vào sản phẩm nhanh chóng và dễ dàng.

---

## Văn Hóa Công Ty

- **Cộng Đồng và Hợp Tác**: Hugging Face đề cao tinh thần cộng đồng, khuyến khích sự hợp tác mở giữa các nhà phát triển, nhà nghiên cứu và các tổ chức trên toàn thế giới.
- **Minh Bạch và Chia Sẻ**: Công ty tạo điều kiện để cộng đồng truy cập, chia sẻ kiến thức, dữ liệu và mô hình AI một cách công khai.
- **Đổi Mới Liên Tục**: Luôn dẫn đầu trong các xu hướng phát triển AI mới nhất với sự hợp tác của các chuyên gia hàng đầu và các đối tác chiến lược tầm cỡ như NVIDIA.

---

## Khách Hàng & Đối Tác

- Cộng đồng machine learning toàn cầu gồm nhà nghiên cứu, kỹ sư AI, developer.
- Các doanh nghiệp công nghệ và tổ chức lớn mong muốn áp dụng AI cho sản phẩm và dịch vụ của mình.
- Hợp tác chiến lược với **NVIDIA**, gia tăng sức mạnh công nghệ AI.

---

## Cơ Hội Nghề Nghiệp

- Hugging Face mở rộng đội ngũ phát triển sản phẩm, kỹ thuật và cộng đồng.
- Môi trường làm việc thân thiện với cộng đồng AI sôi động, cởi mở, khuyến khích sáng tạo và học hỏi liên tục.
- Nhiều vị trí dành cho kỹ sư machine learning, nhà khoa học dữ liệu, kỹ sư phần mềm và các chuyên gia công nghệ khác.

Ứng viên quan tâm có thể tham khảo thông tin và ứng tuyển qua trang web chính thức của công ty.

---

## Kết Luận

Hugging Face là nền tảng và cộng đồng AI hàng đầu dành cho mọi người làm việc với AI. Với sứ mệnh xây dựng tương lai AI bằng cách kết nối và hỗ trợ các nhà phát triển cũng như doanh nghiệp, lựa chọn Hugging Face là lựa chọn chiến lược cho những ai muốn dẫn đầu trong cuộc cách mạng AI toàn cầu.

---

**Website:** [huggingface.co](https://huggingface.co)  
**Cộng đồng:** Discord, Forum, GitHub  
**Hợp tác chiến lược:** NVIDIA

---

*Hãy gia nhập cộng đồng Hugging Face để cùng xây dựng tương lai AI*

## Cuối cùng — một cải tiến nhỏ

Với một điều chỉnh nhỏ, chúng ta có thể đổi để kết quả stream (luồng chảy) về từ OpenAI,
với hiệu ứng typewriter (máy đánh chữ) quen thuộc

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Đang chọn relevant links cho https://huggingface.co bằng cách gọi gpt-5-nano
Tìm thấy 15 relevant links


# Hugging Face — Brochure Tóm Tắt

---

## Giới Thiệu Chung

**Hugging Face** là nền tảng cộng đồng AI hàng đầu, nơi các nhà nghiên cứu, kỹ sư machine learning, và người dùng cuối trên toàn thế giới cùng nhau xây dựng và phát triển tương lai của trí tuệ nhân tạo. Đây là không gian mở để chia sẻ, khám phá, và hợp tác trên hàng triệu mô hình, tập dữ liệu, và ứng dụng ML.

---

## Sản Phẩm & Dịch Vụ

- **Hugging Face Hub**: Nơi lưu trữ hơn 2 triệu mô hình machine learning sẵn sàng sử dụng, bao gồm các lĩnh vực như NLP, computer vision, speech...
  
- **Datasets**: Kho dữ liệu lớn, đa dạng với hơn 500k bộ dataset cho nghiên cứu và triển khai.
  
- **Spaces**: Nền tảng chạy và chia sẻ ứng dụng ML trực tiếp trên cloud.
  
- **Enterprise Solutions**: Đáp ứng nhu cầu doanh nghiệp với các gói hỗ trợ chuyên sâu, inference endpoints, và lưu trữ an toàn.
  
- **HuggingChat**: Giải pháp AI chat tương tác thông minh trên nền tảng mở.

- **Hợp tác với NVIDIA**: Mục tiêu phát triển và tối ưu hệ sinh thái AI với đối tác công nghệ hàng đầu.

---

## Văn Hóa Công Ty

- **Cộng đồng & Hợp tác**: Hugging Face thúc đẩy môi trường làm việc cởi mở, minh bạch, khuyến khích sự cộng tác và chia sẻ kiến thức giữa các nhà khoa học dữ liệu, kỹ sư và người dùng toàn cầu.
  
- **Mở & Đạo đức**: Cam kết xây dựng AI mở, minh bạch và có trách nhiệm với xã hội, tôn trọng các nguyên tắc đạo đức trong phát triển công nghệ.
  
- **Sáng tạo & Tiên phong**: Đội ngũ Hugging Face luôn ở tuyến đầu nghiên cứu và áp dụng các công nghệ máy học hiện đại nhất.

---

## Khách Hàng & Đối Tác

- Các nhà nghiên cứu và kỹ sư machine learning trên toàn thế giới.
- Doanh nghiệp và tổ chức từ các lĩnh vực công nghệ, tài chính, y tế, truyền thông...
- Cộng đồng developer đa dạng với hơn hàng trăm nghìn người dùng thường xuyên.
- Đối tác chiến lược: NVIDIA cùng đồng hành mở rộng tiềm năng AI.

---

## Cơ Hội Nghề Nghiệp

- Hugging Face chào đón những ứng viên đam mê AI, blockchain, phát triển sản phẩm và cộng đồng.
- Môi trường làm việc đa dạng, sáng tạo với các vị trí: Kỹ sư machine learning, kỹ sư phần mềm, nhà khoa học dữ liệu, kỹ sư DevOps, quản lý sản phẩm...
- Tận hưởng văn hóa làm việc năng động, cơ hội học hỏi liên tục và đóng góp trực tiếp cho cộng đồng AI toàn cầu.

---

## Liên Hệ & Tìm Hiểu Thêm

- Website chính thức: [huggingface.co](https://huggingface.co)  
- Cộng đồng Discord và Forum sôi nổi, giúp kết nối và hỗ trợ giữa các thành viên.  
- GitHub: Các nền tảng mã nguồn mở và công cụ phát triển cập nhật thường xuyên.  
- Mạng xã hội: Twitter, LinkedIn để theo dõi các tin tức và sự kiện mới nhất.

---

Hugging Face — Cùng bạn xây dựng tương lai AI mở và bền vững!

In [25]:
# Hãy thử đổi system prompt (prompt hệ thống) sang phiên bản hài hước khi làm Brochure cho Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Đang chọn relevant links cho https://huggingface.co bằng cách gọi gpt-5-nano
Tìm thấy 11 relevant links


# Brochure giới thiệu công ty Hugging Face

---

## Giới thiệu chung

**Hugging Face** là cộng đồng AI tiên phong xây dựng tương lai công nghệ thông minh. Đây là nền tảng hợp tác dành cho cộng đồng máy học (machine learning), nơi mà các kỹ sư, nhà khoa học và người dùng cuối trên toàn thế giới cùng sáng tạo, chia sẻ và phát triển các mô hình, bộ dữ liệu, và ứng dụng AI mã nguồn mở.

Hugging Face không chỉ là một công ty mà còn là một hệ sinh thái giúp mọi người dễ dàng tiếp cận, thử nghiệm và ứng dụng các công nghệ AI tiên tiến để thúc đẩy sự phát triển của trí tuệ nhân tạo một cách minh bạch, mở và có trách nhiệm.

---

## Sản phẩm & Dịch vụ chính

- **Models**: Thư viện hơn 2 triệu mô hình AI đa dạng, cập nhật liên tục cho nhiều ứng dụng khác nhau như xử lý ngôn ngữ tự nhiên, hình ảnh, video,...
- **Datasets**: Hơn 500 nghìn bộ dữ liệu mã nguồn mở để phục vụ việc huấn luyện và thử nghiệm các mô hình AI.
- **Spaces**: Nền tảng chạy ứng dụng AI trực tuyến, cho phép người dùng thử nghiệm các demo và app được xây dựng dựa trên mô hình AI một cách nhanh chóng.
- **Buckets**: Giải pháp lưu trữ dữ liệu và mô hình AI hiệu quả, hỗ trợ xử lý và kiến trúc đa dạng.
- **HuggingChat**: Chatbot AI tương tác thông minh, ứng dụng tiên tiến trong lĩnh vực tạo hội thoại tự nhiên.
- **Enterprise Solutions**: Các gói dịch vụ dành cho doanh nghiệp như hỗ trợ kỹ thuật, endpoint inference, và lưu trữ chuyên biệt.

---

## Văn hóa Công ty (Company Culture)

- **Cộng đồng hợp tác và phát triển**: Hugging Face đặt cộng đồng và sự kết nối sáng tạo làm trung tâm, giúp các thành viên cùng học hỏi, chia sẻ và đóng góp vào công nghệ AI mở.
- **Minh bạch và có trách nhiệm**: Cam kết thúc đẩy AI một cách minh bạch, có đạo đức và bền vững, chú trọng vào phát triển công nghệ vì lợi ích chung.
- **Đa dạng và sáng tạo**: Môi trường làm việc đa dạng, tôn trọng suy nghĩ sáng tạo, và khuyến khích đổi mới liên tục.
- **Học hỏi liên tục**: Cung cấp tài nguyên học tập, học bổng và hỗ trợ nghiên cứu để mọi cá nhân phát triển kỹ năng và kiến thức AI tiên tiến nhất.

---

## Khách hàng & Cộng đồng

- Hugging Face phục vụ hơn hàng trăm nghìn kỹ sư, nhà nghiên cứu, tập đoàn và startups trên toàn thế giới.
- Các tổ chức doanh nghiệp lớn tận dụng nền tảng để triển khai mô hình AI quy mô lớn, tối ưu hóa công việc và phát triển sản phẩm AI.
- Cộng đồng phát triển mở rộng nhanh chóng với hơn 100,000 thành viên hoạt động tích cực trên GitHub, Discord, và các diễn đàn AI.
- Đơn vị hợp tác chiến lược với NVIDIA để thúc đẩy công nghệ AI và xây dựng hệ sinh thái AI toàn diện.

---

## Cơ hội nghề nghiệp (Careers)

- Hugging Face thường xuyên tuyển dụng các vị trí kỹ sư AI, nhà khoa học dữ liệu, kỹ sư phần mềm, chuyên gia UX/UI, và nhiều vị trí liên quan đến công nghệ AI.
- Môi trường làm việc linh hoạt, hỗ trợ phát triển cá nhân và thăng tiến.
- Cơ hội làm việc với các chuyên gia đầu ngành, tham gia dự án AI quy mô lớn có ảnh hưởng toàn cầu.
- Công ty cung cấp các chế độ đãi ngộ cạnh tranh, chú trọng tới sức khỏe và cân bằng cuộc sống nhân viên.

---

## Liên hệ & Tham khảo

- Website: [huggingface.co](https://huggingface.co)
- Cộng đồng: Discord, Forum, GitHub
- Đọc blog, tài liệu & xem các bản cập nhật mới nhất liên quan đến AI và machine learning
- Khám phá tài nguyên miễn phí và các bộ sưu tập model, dữ liệu [tại đây](https://huggingface.co/models)

---

**Hugging Face – Nơi hội tụ và phát triển trí tuệ nhân tạo mở, vì một tương lai công nghệ sáng tạo và bền vững.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng kinh doanh</h2>
            <span style="color:#181;">Trong bài tập này chúng ta mở rộng code Day 1 để thực hiện nhiều lời gọi LLM, rồi sinh ra một document (tài liệu).

Đây có lẽ là ví dụ đầu tiên về Agentic AI design patterns (mẫu thiết kế AI dạng agent), vì chúng ta kết hợp nhiều lời gọi LLM. Điều này sẽ xuất hiện nhiều hơn ở Week 2, rồi chúng ta sẽ quay lại Agentic AI quy mô lớn ở Week 8 khi xây một giải pháp Agent tự chủ hoàn chỉnh.

Sinh nội dung theo cách này là một trong những Use Cases (tình huống sử dụng) phổ biến nhất. Giống summarization (tóm tắt), có thể áp dụng cho bất kỳ lĩnh vực kinh doanh nào. Viết nội dung marketing, sinh hướng dẫn sản phẩm từ một spec (đặc tả), tạo email cá nhân hóa, và còn nhiều nữa. Hãy khám phá cách áp dụng content generation (sinh nội dung) vào công việc của bạn, và thử làm một proof-of-concept prototype (bản mẫu chứng minh ý tưởng). Xem học viên khác đã làm gì trong thư mục community-contributions — rất nhiều dự án giá trị — thật điên rồ!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Trước khi sang Week 2 (rất vui đó)</h2>
            <span style="color:#900;">Hãy xem notebook EXERCISE của week1 cho thử thách cuối tuần 1. Việc này sẽ cho bạn luyện tập thiết yếu khi làm việc với Frontier APIs (API các mô hình tiên phong), và chuẩn bị tốt cho Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc lại 3 tài nguyên hữu ích</h2>
            <span style="color:#f71;">1. Tài nguyên khóa học có <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">tại đây.</a><br/>
            2. Tôi có LinkedIn <a href="https://www.linkedin.com/in/eddonner/">tại đây</a> và rất thích kết nối với người đang học khóa này!<br/>
            3. Tôi đang thử X/Twitter tại <a href="https://x.com/edwarddonner">@edwarddonner<a> và mong mọi người chỉ cho tôi cách dùng..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Cuối cùng! Tôi có một lời nhờ đặc biệt với bạn</h2>
            <span style="color:#090;">
                Biên tập viên của tôi nói rằng việc học viên đánh giá khóa học trên Udemy tạo ra sự khác biệt RẤT LỚN — đó là một trong những cách chính để Udemy quyết định có hiện khóa này cho người khác không. Nếu bạn dành được một phút để đánh giá, tôi sẽ biết ơn vô cùng! Và dù sao đi nữa — luôn hãy liên hệ ed@edwarddonner.com nếu tôi có thể giúp bất cứ lúc nào.
            </span>
        </td>
    </tr>
</table>